In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

In [2]:
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor 

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = OneHotEncoder(sparse_output=False)

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [7]:
alphas_alt = [14.5, 14.6, 14.7, 14.8, 14.9, 15, 15.1, 15.2, 15.3, 15.4, 15.5]
alphas2 = [5e-05, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008]
e_alphas = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007]
e_l1ratio = [0.8, 0.85, 0.9, 0.95, 0.99, 1]

In [8]:
from sklearn.model_selection import GridSearchCV 
from sklearn.ensemble import RandomForestRegressor


param_grid = { 
    'n_estimators': [25, 50, 100, 150], 
    'max_features': ['sqrt', 'log2', None], 
    'max_depth': [3, 6, 9], 
    'max_leaf_nodes': [3, 6, 9], 
} 


svr=SVR()
grid = GridSearchCV(, param_grid, refit = True, verbose = 3) 

grid.fit(train_x, train_y) 

Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV 1/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=25;, score=0.062 total time=   0.2s
[CV 2/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=25;, score=0.060 total time=   0.2s
[CV 3/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=25;, score=0.060 total time=   0.2s
[CV 4/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=25;, score=0.059 total time=   0.2s
[CV 5/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=25;, score=0.059 total time=   0.2s
[CV 1/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=50;, score=0.068 total time=   0.4s
[CV 2/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=50;, score=0.055 total time=   0.4s
[CV 3/5] END max_depth=3, max_features=sqrt, max_leaf_nodes=3, n_estimators=50;, score=0.059 total time=   0.4s
[CV 4/5] END max_depth=3, max_features=sq

GridSearchCV(estimator=RandomForestRegressor(),
             param_grid={'max_depth': [3, 6, 9],
                         'max_features': ['sqrt', 'log2', None],
                         'max_leaf_nodes': [3, 6, 9],
                         'n_estimators': [25, 50, 100, 150]},
             verbose=3)

In [9]:
print(grid.best_params_) 
  
print(grid.best_estimator_) 

{'max_depth': 9, 'max_features': None, 'max_leaf_nodes': 9, 'n_estimators': 100}
RandomForestRegressor(max_depth=9, max_features=None, max_leaf_nodes=9)


In [10]:
randomforest=make_pipeline(RobustScaler(), 
                           RandomForestRegressor(max_depth=9, max_features=None, max_leaf_nodes=9, n_estimators=100))
svr = make_pipeline(RobustScaler(), 
                    SVR(kernel='rbf', C= 1, epsilon= 0.1, gamma=0.1))

In [11]:
gbr = GradientBoostingRegressor(n_estimators=3000, 
                                learning_rate=0.05, 
                                max_depth=4, 
                                max_features='sqrt', 
                                min_samples_leaf=15, 
                                min_samples_split=10, 
                                loss='huber', 
                                random_state =42)    

lightgbm = LGBMRegressor(objective='regression', 
                         num_leaves=4,
                         learning_rate=0.01, 
                         n_estimators=5000,
                         max_bin=200, 
                         verbose=-1)

AdaBoost = AdaBoostRegressor(learning_rate=1, 
                             n_estimators=100)

xgboost = XGBRegressor(device="cuda",
                       max_depth=3,  
                       colsample_bytree=0.5, 
                       subsample=0.8, 
                       n_estimators=10_000,  
                       learning_rate=0.01, 
                       eval_metric="mae",
                       early_stopping_rounds=200,
                       objective='reg:logistic',
                        #eval_metric='cox-nloglik',
                        #objective='survival:cox',
                       enable_categorical=True,
                       min_child_weight=5
                    )

catboost = CatBoostRegressor(loss_function='RMSE',
                             n_estimators=10000,
                             learning_rate=0.1)

In [12]:
model = StackingRegressor(estimators=[('ridge', randomforest), ('lasso', lasso), 
                                      ('elasticnet', elasticnet), 
                                      ('svr', svr), ('gbr', gbr),
                                      ('xgboost', xgboost), ('lightgbm', lightgbm),
                                      ('AdaBoost', AdaBoost)],
                                      final_estimator=xgboost)

NameError: name 'ridge' is not defined

In [ ]:
model.fit(train_x, train_y)

In [ ]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

In [ ]:

test_predictions = model.predict(test)

In [ ]:
test_predictions

In [ ]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

In [ ]:
submission.to_csv('submission.csv', index = False)